# 高购买意向用户触达策略 A/B 实验设计

## 1. 实验背景

LightGBM 能够根据会话前 5 次行为识别高购买意向会话。在时间外测试集中，预测分数最高的 10% 会话实际购买率为 22.30%，覆盖了 34.11% 的后续购买会话。

模型只能说明哪些会话更可能购买，不能证明展示推荐、提醒或优惠券能够提升购买率。因此，需要通过随机 A/B 实验验证触达策略的增量效果。

## 2. 待验证策略

当用户完成会话前 5 次行为、尚未购买，且模型预测分数进入高意向人群时，触发个性化商品推荐。

- 对照组：维持原有页面和推荐策略；
- 实验组：展示基于用户当前浏览商品和品类生成的个性化推荐模块；
- 两组除该推荐模块外保持一致。

第一版优先测试推荐模块，不直接测试优惠券。优惠券会引入补贴成本，并且可能把原本就会购买的用户变成需要补贴才能购买的用户。

## 3. 实验对象与随机化

### 实验对象

满足以下条件的用户进入实验：

1. 当前会话至少完成 5 次行为；
2. 前 5 次行为中尚未购买；
3. LightGBM 预测分数位于目标触达范围；
4. 用户在实验期间未参加其他互斥实验。

### 随机化单位

以 `user_id` 作为随机化单位，而不是 `user_session`。

同一用户可能拥有多个会话。如果按会话随机分组，同一用户可能同时进入实验组和对照组，造成实验污染。按用户随机分组可以确保同一用户在实验期间始终看到同一种策略。

## 4. 实验指标

### 核心指标

用户购买转化率：

```text
实验期间发生购买的用户数 ÷ 进入实验的用户数

次要指标
- 加购率；
- 人均购买金额；
- 人均收入；
- 推荐模块点击率。

护栏指标
- 页面退出率；
- 推荐模块关闭率；
- 负反馈率；
- 退款率或取消率；
- 延迟和页面加载失败率。

实验不能只追求购买率提升。如果购买率提高，但退款率、负反馈或页面退出率明显上升，也不应直接全量上线。
实验决策原则
只有在核心指标统计显著、效果量达到业务要求，且护栏指标没有明显恶化时，才考虑扩大流量或全量上线。


## 5. 设定样本量参数

测试集 Top 10% 会话的实际购买率是：

```text
22.30%

## 5. 样本量假设

根据测试集结果，高意向目标人群的历史购买率约为 22.30%。

本实验暂设：

- 对照组基线转化率：22.30%；
- 目标相对提升：10%；
- 实验组目标转化率：24.53%；
- 显著性水平：0.05；
- 统计功效：0.80；
- 实验组和对照组按 1:1 分流。

样本量估算仅用于实验设计，不代表真实实验结果。

In [1]:
#计算每组样本量
try:
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize
    print("statsmodels 已安装")
except ModuleNotFoundError:
    print("当前环境没有 statsmodels")

statsmodels 已安装


In [ ]:
%pip install statsmodels

In [2]:
#计算样本量
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline_rate = 0.2230
relative_mde = 0.10
treatment_rate = baseline_rate * (1 + relative_mde)

alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(
    baseline_rate,
    treatment_rate
)

sample_size_per_group = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided"
)

sample_size_per_group = int(sample_size_per_group + 0.5)
total_sample_size = sample_size_per_group * 2

print("对照组基线转化率：", baseline_rate)
print("实验组目标转化率：", round(treatment_rate, 4))
print("目标相对提升：", relative_mde * 100, "%")
print("每组所需样本量：", sample_size_per_group)
print("总样本量：", total_sample_size)

对照组基线转化率： 0.223
实验组目标转化率： 0.2453
目标相对提升： 10.0 %
每组所需样本量： 5658
总样本量： 11316


In [3]:
#加入实验流失冗余
dropout_rate = 0.10

adjusted_sample_size_per_group = int(
    sample_size_per_group
    / (1 - dropout_rate)
    + 0.5
)

adjusted_total_sample_size = (
    adjusted_sample_size_per_group * 2
)

print(
    "考虑 10% 流失后，每组所需样本量：",
    adjusted_sample_size_per_group
)

print(
    "考虑 10% 流失后，总样本量：",
    adjusted_total_sample_size
)

考虑 10% 流失后，每组所需样本量： 6287
考虑 10% 流失后，总样本量： 12574


### 样本量计算结果

以高意向目标人群历史购买率 22.30% 作为对照组基线，并设定实验策略需要检测到至少 10% 的相对提升，即实验组目标购买率为 24.53%。

在显著性水平 0.05、统计功效 0.80、双侧检验和实验组与对照组 1:1 分流的条件下：

- 每组理论上需要 5,658 名用户；
- 两组共需要 11,316 名用户；
- 考虑 10% 的用户流失、埋点缺失或实验排除后，每组计划招募 6,287 名用户；
- 调整后的总样本量为 12,574 名用户。

本结果是实验设计阶段的样本量估算，不代表已经实施实验，也不代表推荐策略已经带来转化提升。

## 6. 实验执行流程

1. 确定满足条件的目标用户；
2. 在用户级别进行随机分组；
3. 检查实验组和对照组的样本量是否接近；
4. 检查实验前关键指标是否平衡；
5. 实验期间保持用户分组不变；
6. 到达预先设定的实验周期和样本量后再分析；
7. 计算购买转化率差异、置信区间和 p-value；
8. 进行预先定义的用户分层分析；
9. 检查护栏指标；
10. 根据统计显著性和业务收益决定继续、灰度或停止。

### 实验质量检查

- SRM：实验组和对照组流量比例是否偏离预设比例；
- 重复入组：同一用户是否进入多个组；
- 埋点完整性：曝光、点击和购买事件是否正常记录；
- 基线平衡：实验前行为和历史购买率是否接近；
- 实验污染：对照组是否看到了实验策略；
- 提前停止：不能因为中途偶然显著就提前结束实验。

## 7. 观察性数据限制

当前 Kaggle 数据没有实验组、对照组和策略曝光字段，因此不能使用历史数据直接计算推荐策略的因果效果，也不能伪造 A/B 实验结果。

本节提供的是一个可落地的线上实验设计：

- 模型用于筛选高购买意向用户；
- 随机实验用于验证推荐策略是否带来增量购买；
- 购买率差异只有在随机分组、实验质量通过检查后，才可以解释为策略的因果效果。

当前模型的测试集结果只能说明高意向会话具有更高的自然购买概率，不能说明推荐模块一定能够提高购买率。